# 1 – Dati Spaziali: caricamento e preparazione

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/1_spatial_data.R`

- Caricamento shapefile (dati areali e puntuali)
- Conversione CSV → GeoDataFrame
- Aggregazione su griglia esagonale H3
- Salvataggio in formato GeoParquet

In [ ]:
!pip install -q geopandas pyarrow h3

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from shapely.geometry import Polygon
import h3
import warnings
warnings.filterwarnings('ignore')

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## Dati areali – Tanzania DHS 2022

In [ ]:
tanzania = gpd.read_parquet("tanzania.parquet")
print(tanzania.crs)
tanzania.head()

## Dati puntuali – King County housing

In [ ]:
kc = gpd.read_parquet("kc_house.parquet")
print(kc.crs)
kc.head()

### Da CSV a GeoDataFrame

In [ ]:
!wget -q -nc https://github.com/vincnardelli/spec/raw/main/data_raw/kingcounty/kc_house_data.csv

kc_df = pd.read_csv("kc_house_data.csv")
kc_sf = gpd.GeoDataFrame(
    kc_df,
    geometry=gpd.points_from_xy(kc_df["long"], kc_df["lat"]),
    crs="EPSG:4326"
)
print(kc_sf.crs)
kc_sf.head()

## Aggregazione punti → griglia H3

In [ ]:
kc["h3"] = kc.geometry.apply(lambda p: h3.latlng_to_cell(p.y, p.x, 8))

kc_grid = (
    kc.drop(columns="geometry")
    .groupby("h3", as_index=False)
    .agg(price=("price", "mean"))
)

def h3_to_polygon(h):
    boundary = h3.cell_to_boundary(h)
    return Polygon([(lng, lat) for lat, lng in boundary])

kc_grid = gpd.GeoDataFrame(
    kc_grid,
    geometry=kc_grid["h3"].apply(h3_to_polygon),
    crs="EPSG:4326"
)
kc_grid.head()